# Preparación de vectores PM2.5/PM10

Cada vector representa dos concentraciones del mismo informe/evaluación, estación, fecha, hora, periodo y fin de medición. Se excluyen claves ambiguas con varias mediciones del mismo contaminante. La agrupación conserva datos faltantes fuera de los pares en lugar de imputar o replicar valores.


In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at <DRIVE_MOUNT>


In [2]:
from pathlib import Path
import json
import pandas as pd
from datetime import datetime,timezone
from IPython.display import display
DATA_DIR=Path('/content/drive/MyDrive/Concurrente')
manifest=json.loads((DATA_DIR/'Procesado_PM'/'ultima_ejecucion.json').read_text())
df=pd.read_parquet(manifest['parquet'])
assert set(df.contaminante.unique()) <= {'PM10','PM2.5'}
assert len(df)==manifest['filas']
print('Mediciones reales limpias:',len(df))
display(df.head())


Mediciones reales limpias: 1135970


,informe,evaluacion,punto,este,norte,zona,datum,altitud,periodo,fecha,hora,hora_original,fecha_fin_original,hora_fin_original,contaminante,concentracion,unidad,ventana_2022_2024
0,REAS-009-2023-STEC,Calidad del aire en el ámbito de influencia de...,CA-HUA-01,504395.0,9456762.0,17,WGS84,32.0,1 hora,2022-02-03,None,0.625,-,-,PM2.5,22.8,µg/m³,True
1,REAS-009-2023-STEC,Calidad del aire en el ámbito de influencia de...,CA-HUA-01,504395.0,9456762.0,17,WGS84,32.0,1 hora,2022-02-03,None,0.70833333333333337,-,-,PM2.5,20.6,µg/m³,True
2,REAS-009-2023-STEC,Calidad del aire en el ámbito de influencia de...,CA-HUA-01,504395.0,9456762.0,17,WGS84,32.0,1 hora,2022-02-03,None,0.75,-,-,PM2.5,14.0,µg/m³,True
3,REAS-009-2023-STEC,Calidad del aire en el ámbito de influencia de...,CA-HUA-01,504395.0,9456762.0,17,WGS84,32.0,1 hora,2022-02-03,None,0.83333333333333337,-,-,PM2.5,6.3,µg/m³,True
4,REAS-009-2023-STEC,Calidad del aire en el ámbito de influencia de...,CA-HUA-01,504395.0,9456762.0,17,WGS84,32.0,1 hora,2022-02-04,None,0.125,-,-,PM2.5,7.3,µg/m³,True


## Definición de la observación y control de réplicas

Se conservan las claves espaciales y temporales. La hora y periodo deben estar presentes; no se mezclan medias móviles y lecturas horarias. Se revisarán métodos y representatividad antes de entrenar. No se asigna significado numérico a los códigos de estación o contaminante.


In [3]:
keys=['informe','evaluacion','punto','este','norte','zona','datum','fecha','hora','periodo','fecha_fin_original','hora_fin_original']
work=df.copy()
for col in keys:
    if work[col].dtype=='object': work[col]=work[col].astype('string').str.strip()
valid=work[keys].notna().all(axis=1)
for col in ['informe','punto','datum','periodo','fecha_fin_original','hora_fin_original']:
    valid &= ~work[col].astype('string').isin(['','-','N.I.','NA'])
work=work[valid].copy()
counts=work.groupby(keys+['contaminante'],dropna=False).size().rename('n')
ambiguous=counts[counts>1].reset_index()[keys].drop_duplicates()
if len(ambiguous):
    work=work.merge(ambiguous.assign(ambigua=True),on=keys,how='left')
    work=work[work.ambigua.isna()].drop(columns='ambigua')
print('Claves ambiguas excluidas:',len(ambiguous))
print('Mediciones con clave temporal completa y sin réplicas:',len(work))


Claves ambiguas excluidas: 0
Mediciones con clave temporal completa y sin réplicas: 23206


## Pivotado sin promediar

Se utiliza pivot, sin aggfunc=mean. Solo se conservan ambos componentes PM10 y PM2.5.


In [4]:
if len(work):
    wide=work.pivot(index=keys,columns='contaminante',values='concentracion').reindex(columns=['PM10','PM2.5'])
    pairs=wide.dropna(subset=['PM10','PM2.5']).reset_index()
else:
    pairs=pd.DataFrame(columns=keys+['PM10','PM2.5'])
print('Pares reales PM10/PM2.5 disponibles:',len(pairs))
display(pairs.head())



Pares reales PM10/PM2.5 disponibles: 10730


contaminante,informe,evaluacion,punto,este,norte,zona,datum,fecha,hora,periodo,fecha_fin_original,hora_fin_original,PM10,PM2.5
0,REAS-002-2024-STEC,calidad del aire en el ámbito de influencia de...,CA-HU-01,290348.0,8674301.0,18,WSG84,2023-01-01,00:00:00,24 horas,44927,23:00:00,124.5,67.3
1,REAS-002-2024-STEC,calidad del aire en el ámbito de influencia de...,CA-HU-01,290348.0,8674301.0,18,WSG84,2023-01-02,00:00:00,24 horas,44928,23:00:00,166.4,69.6
2,REAS-002-2024-STEC,calidad del aire en el ámbito de influencia de...,CA-HU-01,290348.0,8674301.0,18,WSG84,2023-01-03,00:00:00,24 horas,44929,23:00:00,138.1,73.7
3,REAS-002-2024-STEC,calidad del aire en el ámbito de influencia de...,CA-HU-01,290348.0,8674301.0,18,WSG84,2023-01-04,00:00:00,24 horas,44930,23:00:00,142.5,69.0
4,REAS-002-2024-STEC,calidad del aire en el ámbito de influencia de...,CA-HU-01,290348.0,8674301.0,18,WSG84,2023-01-05,00:00:00,24 horas,44931,23:00:00,284.3,97.2


## Exportación y entrega al modelo

El CSV conserva concentraciones físicas y metadatos.


In [5]:
out=DATA_DIR/'Vectores_PM'/datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
out.mkdir(parents=True,exist_ok=False)
pairs.to_csv(out/'vectores_PM10_PM25.csv',index=False)
report={'mediciones_limpias':len(df),'mediciones_con_clave_validada':len(work),'claves_ambiguas':len(ambiguous),'pares_reales':len(pairs),'orden_features':['PM10','PM2.5'],'estado':'PARES DISPONIBLES PARA REVISIÓN' if len(pairs) else 'SIN PARES: REVISAR REPRESENTACIÓN','origen':manifest}
(out/'resumen_vectores.json').write_text(json.dumps(report,indent=2,ensure_ascii=False))
print(report['estado'])
print('Resultados:',out)


PARES DISPONIBLES PARA REVISIÓN
Resultados: <DATA_DIR>/Vectores_PM/20261104T215328_705038Z
